# Koe - Sign Language Recognition Model Training
### Multi-Model Ensemble for Real-Time Gesture Recognition
**Author:** Yusra Batool | **Project:** Koe Accessibility Platform

This notebook trains 5 models on 4 datasets and exports a final ensemble as TFLite files.

**Datasets required:**
- Google - Isolated Sign Language Recognition (Competition)
- Google - American Sign Language Fingerspelling (Competition)
- Sign Language MNIST (Dataset)
- WLASL World Level American Sign Language (Dataset)

## Cell 1: Imports and Environment Check

In [ ]:
import numpy as np
import pandas as pd
import os
import json
import glob
import cv2
import pickle
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

print('TensorFlow:', tf.__version__)
print('GPUs:', tf.config.list_physical_devices('GPU'))
print()
print('Input directories:')
for d in os.listdir('/kaggle/input'):
    print(' -', d)

# Seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

OUTPUT_DIR = '/kaggle/working/koe_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print()
print('Output directory ready:', OUTPUT_DIR)

## Cell 2: Define Koe Sign Vocabulary (30 Core Signs)

In [ ]:
# These are the 30 signs Koe will recognize in the prototype
# Mapped to their ASL gloss labels from the Google dataset
KOE_SIGNS = [
    'help', 'pain', 'doctor', 'call', 'stop',
    'water', 'food', 'medicine', 'yes', 'no',
    'please', 'thank you', 'sorry', 'repeat', 'okay',
    'scared', 'happy', 'home', 'come', 'go',
    'me', 'you', 'family', 'friend', 'nurse',
    'toilet', 'rest', 'confused', 'here', 'not'
]

# ASL gloss labels as they appear in the Google dataset
ASL_GLOSS_MAP = {
    'help': 'help', 'pain': 'pain', 'doctor': 'doctor',
    'call': 'call', 'stop': 'stop', 'water': 'water',
    'food': 'food', 'medicine': 'medicine', 'yes': 'yes',
    'no': 'no', 'please': 'please', 'thank you': 'thank-you',
    'sorry': 'sorry', 'repeat': 'again', 'okay': 'fine',
    'scared': 'scared', 'happy': 'happy', 'home': 'home',
    'come': 'come', 'go': 'go', 'me': 'me',
    'you': 'you', 'family': 'family', 'friend': 'friend',
    'nurse': 'nurse', 'toilet': 'bathroom', 'rest': 'rest',
    'confused': 'confused', 'here': 'here', 'not': 'not'
}

NUM_CLASSES = len(KOE_SIGNS)
print(f'Koe vocabulary: {NUM_CLASSES} signs')
print('Signs:', KOE_SIGNS)

## Cell 3: Load Google ASL Signs Dataset (Landmarks)

In [ ]:
# Exact paths confirmed
ASL_SIGNS_PATH = '/kaggle/input/competitions/asl-signs'
ASL_FINGER_PATH = '/kaggle/input/competitions/asl-fingerspelling'
MNIST_PATH = '/kaggle/input/datasets/datamunge/sign-language-mnist'
WLASL_PATH = '/kaggle/input/datasets/risangbaskoro/wlasl-processed'

print('ASL Signs files:', os.listdir(ASL_SIGNS_PATH))
print('ASL Finger files:', os.listdir(ASL_FINGER_PATH))
print('MNIST files:', os.listdir(MNIST_PATH))
print('WLASL files:', os.listdir(WLASL_PATH))

def load_landmark_file(path):
    try:
        df = pd.read_parquet(path)
        x_cols = [c for c in df.columns if c.startswith('x_')]
        y_cols = [c for c in df.columns if c.startswith('y_')]
        z_cols = [c for c in df.columns if c.startswith('z_')]
        if len(x_cols) > 0:
            row = df[x_cols + y_cols + z_cols].mean()
        else:
            row = df.select_dtypes(include=[np.number]).mean()
        features = row.values[:63].astype(np.float32)
        if len(features) < 63:
            features = np.pad(features, (0, 63 - len(features)))
        features = np.nan_to_num(features, nan=0.0)
        return features
    except:
        return None

# Load ASL Signs
asl_landmarks = []
asl_labels = []

train_df = pd.read_csv(f'{ASL_SIGNS_PATH}/train.csv')
print('ASL Signs CSV shape:', train_df.shape)
print('Columns:', train_df.columns.tolist())
print('Sample signs:', train_df['sign'].unique()[:10])

gloss_values = list(ASL_GLOSS_MAP.values())
filtered_df = train_df[train_df['sign'].isin(gloss_values + KOE_SIGNS)]
if len(filtered_df) == 0:
    print('Koe signs not found directly, using top 30 signs')
    top_signs = train_df['sign'].value_counts().head(30).index.tolist()
    filtered_df = train_df[train_df['sign'].isin(top_signs)]

print(f'Filtered: {len(filtered_df)} samples for {filtered_df["sign"].nunique()} signs')

landmark_base = f'{ASL_SIGNS_PATH}/train_landmark_files'
loaded = 0
for _, row in filtered_df.iterrows():
    path_val = str(row.get('path', row.get('sequence_id', '')))
    full_path = os.path.join(landmark_base, path_val)
    if not full_path.endswith('.parquet'):
        full_path += '.parquet'
    features = load_landmark_file(full_path)
    if features is not None:
        asl_landmarks.append(features)
        asl_labels.append(row['sign'])
        loaded += 1
    if loaded % 1000 == 0 and loaded > 0:
        print(f'  Loaded {loaded} files...')
    if loaded >= 5000:
        break

print(f'ASL Signs loaded: {len(asl_landmarks)} samples')


## Cell 4: Load Sign Language MNIST Dataset

In [ ]:
mnist_landmarks = []
mnist_labels = []

LETTER_SIGNS = ['A','B','C','D','E','F','G','H','I','K',
                'L','M','N','O','P','Q','R','S','T','U',
                'V','W','X','Y']

mnist_files = os.listdir(MNIST_PATH)
print('MNIST files:', mnist_files)

train_file = [f for f in mnist_files if 'train' in f.lower() and f.endswith('.csv')]
if train_file:
    mnist_df = pd.read_csv(f'{MNIST_PATH}/{train_file[0]}')
    print('MNIST shape:', mnist_df.shape)
    label_col = mnist_df.columns[0]
    pixel_cols = mnist_df.columns[1:]
    labels_raw = mnist_df[label_col].values
    images = mnist_df[pixel_cols].values.reshape(-1, 28, 28).astype(np.float32) / 255.0
    for i in range(len(images)):
        img = images[i]
        features = np.concatenate([
            np.mean(img, axis=0)[:21],
            np.mean(img, axis=1)[:21],
            np.std(img, axis=0)[:21]
        ]).astype(np.float32)
        features = features[:63]
        if len(features) < 63:
            features = np.pad(features, (0, 63 - len(features)))
        letter_idx = int(labels_raw[i])
        if letter_idx < len(LETTER_SIGNS):
            mnist_landmarks.append(features)
            mnist_labels.append(f'letter_{LETTER_SIGNS[letter_idx]}')
    print(f'MNIST loaded: {len(mnist_landmarks)} samples')
else:
    print('MNIST train CSV not found, files:', mnist_files)


## Cell 5: Load ASL Fingerspelling Dataset

In [ ]:
finger_landmarks = []
finger_labels = []

print('Fingerspelling files:', os.listdir(ASL_FINGER_PATH))

finger_df = pd.read_csv(f'{ASL_FINGER_PATH}/train.csv')
print('Fingerspelling CSV shape:', finger_df.shape)
print('Columns:', finger_df.columns.tolist())

landmark_base_finger = f'{ASL_FINGER_PATH}/train_landmarks'
loaded = 0

for _, row in finger_df.iterrows():
    path_val = str(row.get('path', row.get('file', '')))
    full_path = os.path.join(landmark_base_finger, path_val)
    if not full_path.endswith('.parquet'):
        full_path += '.parquet'
    if os.path.exists(full_path):
        features = load_landmark_file(full_path)
        if features is not None:
            phrase = str(row.get('phrase', row.get('letter', 'a')))
            letter = phrase[0].upper() if phrase else 'A'
            if letter.isalpha():
                finger_landmarks.append(features)
                finger_labels.append(f'letter_{letter}')
                loaded += 1
    if loaded % 1000 == 0 and loaded > 0:
        print(f'  Loaded {loaded} fingerspelling files...')
    if loaded >= 3000:
        break

print(f'Fingerspelling loaded: {len(finger_landmarks)} samples')


## Cell 6: Combine All Datasets and Prepare Features

In [ ]:
# Combine all loaded data
all_features = []
all_labels = []

# Add ASL Signs data
if asl_landmarks:
    for feat, label in zip(asl_landmarks, asl_labels):
        feat = np.array(feat, dtype=np.float32)
        feat = feat[:63] if len(feat) >= 63 else np.pad(feat, (0, max(0, 63-len(feat))))
        all_features.append(feat)
        all_labels.append(label)
    print(f'Added {len(asl_landmarks)} ASL Signs samples')

# Add MNIST data
if mnist_landmarks:
    all_features.extend(mnist_landmarks)
    all_labels.extend(mnist_labels)
    print(f'Added {len(mnist_landmarks)} MNIST samples')

# Add Fingerspelling data
if finger_landmarks:
    all_features.extend(finger_landmarks)
    all_labels.extend(finger_labels)
    print(f'Added {len(finger_landmarks)} Fingerspelling samples')

print(f'\nTotal combined samples: {len(all_features)}')

# Convert to numpy arrays
X = np.array(all_features, dtype=np.float32)
print('Feature shape:', X.shape)

# Normalize features
X = np.nan_to_num(X, nan=0.0, posinf=1.0, neginf=-1.0)
X_mean = X.mean(axis=0)
X_std = X.std(axis=0) + 1e-8
X_norm = (X - X_mean) / X_std

# Encode labels
le = LabelEncoder()
y = le.fit_transform(all_labels)
NUM_ACTUAL_CLASSES = len(le.classes_)

print(f'Number of classes: {NUM_ACTUAL_CLASSES}')
print('Classes:', le.classes_[:10], '...')

# Save label encoder
with open(f'{OUTPUT_DIR}/label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# Save normalization stats
np.save(f'{OUTPUT_DIR}/X_mean.npy', X_mean)
np.save(f'{OUTPUT_DIR}/X_std.npy', X_std)

# Train/val/test split
X_temp, X_test, y_temp, y_test = train_test_split(
    X_norm, y, test_size=0.1, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.11, random_state=42, stratify=y_temp
)

print(f'\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

# One-hot encode
y_train_oh = keras.utils.to_categorical(y_train, NUM_ACTUAL_CLASSES)
y_val_oh = keras.utils.to_categorical(y_val, NUM_ACTUAL_CLASSES)
y_test_oh = keras.utils.to_categorical(y_test, NUM_ACTUAL_CLASSES)

print('Data preparation complete')

## Cell 7: Model 1 - MLP Static Classifier

In [ ]:
def build_mlp(input_dim, num_classes):
    inputs = keras.Input(shape=(input_dim,), name='landmarks')
    x = layers.Dense(512, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    model = Model(inputs, outputs, name='koe_mlp')
    return model

mlp_model = build_mlp(63, NUM_ACTUAL_CLASSES)
mlp_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
mlp_model.summary()

callbacks_mlp = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ModelCheckpoint(f'{OUTPUT_DIR}/mlp_best.keras', monitor='val_accuracy',
                   save_best_only=True, verbose=0)
]

print('Training MLP model...')
history_mlp = mlp_model.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=80,
    batch_size=128,
    callbacks=callbacks_mlp,
    verbose=1
)

mlp_test_loss, mlp_test_acc = mlp_model.evaluate(X_test, y_test_oh, verbose=0)
print(f'MLP Test Accuracy: {mlp_test_acc:.4f}')

# Save full model
mlp_model.save(f'{OUTPUT_DIR}/mlp_model.keras')
print('MLP model saved')

## Cell 8: Model 2 - CNN on Landmark Heatmaps

In [ ]:
def landmarks_to_heatmap(landmarks, size=64):
    """Convert 63 landmark features to a 64x64 heatmap image."""
    img = np.zeros((size, size), dtype=np.float32)
    n_points = len(landmarks) // 3
    for i in range(n_points):
        x = landmarks[i * 3]
        y = landmarks[i * 3 + 1]
        # Normalize to image coordinates
        px = int(np.clip((x + 1) / 2 * (size - 1), 0, size - 1))
        py = int(np.clip((y + 1) / 2 * (size - 1), 0, size - 1))
        # Draw gaussian blob
        for dx in range(-2, 3):
            for dy in range(-2, 3):
                nx, ny = px + dx, py + dy
                if 0 <= nx < size and 0 <= ny < size:
                    dist = dx**2 + dy**2
                    img[ny, nx] = max(img[ny, nx], np.exp(-dist / 2.0))
    return img

print('Converting landmarks to heatmaps...')
X_heatmap_train = np.array([landmarks_to_heatmap(x) for x in X_train])
X_heatmap_val = np.array([landmarks_to_heatmap(x) for x in X_val])
X_heatmap_test = np.array([landmarks_to_heatmap(x) for x in X_test])

X_heatmap_train = X_heatmap_train[..., np.newaxis]
X_heatmap_val = X_heatmap_val[..., np.newaxis]
X_heatmap_test = X_heatmap_test[..., np.newaxis]
print('Heatmap shape:', X_heatmap_train.shape)

def build_cnn(num_classes):
    inputs = keras.Input(shape=(64, 64, 1), name='heatmap')
    x = layers.Conv2D(32, 3, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(256, 3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    model = Model(inputs, outputs, name='koe_cnn')
    return model

cnn_model = build_cnn(NUM_ACTUAL_CLASSES)
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
cnn_model.summary()

callbacks_cnn = [
    EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
    ModelCheckpoint(f'{OUTPUT_DIR}/cnn_best.keras', monitor='val_accuracy',
                   save_best_only=True, verbose=0)
]

print('Training CNN model...')
history_cnn = cnn_model.fit(
    X_heatmap_train, y_train_oh,
    validation_data=(X_heatmap_val, y_val_oh),
    epochs=60,
    batch_size=64,
    callbacks=callbacks_cnn,
    verbose=1
)

cnn_test_loss, cnn_test_acc = cnn_model.evaluate(X_heatmap_test, y_test_oh, verbose=0)
print(f'CNN Test Accuracy: {cnn_test_acc:.4f}')
cnn_model.save(f'{OUTPUT_DIR}/cnn_model.keras')
print('CNN model saved')

## Cell 9: Prepare Sequence Data for LSTM and Transformer

In [ ]:
# For LSTM and Transformer we need sequence data
# We simulate sequences by applying temporal augmentation to static landmarks
# In production this will use actual video frame sequences from WLASL

SEQ_LEN = 30  # 30 frames per sequence

def create_sequences_from_static(X, y, seq_len=30):
    """Create pseudo-sequences from static landmarks with augmentation."""
    sequences = []
    labels = []
    for i in range(len(X)):
        landmark = X[i]
        seq = []
        for t in range(seq_len):
            # Add progressive noise to simulate motion
            noise_scale = 0.02 * (t / seq_len)
            noise = np.random.normal(0, noise_scale, landmark.shape)
            frame = np.clip(landmark + noise, -3, 3).astype(np.float32)
            seq.append(frame)
        sequences.append(np.array(seq))
        labels.append(y[i])
    return np.array(sequences), np.array(labels)

print('Creating sequence data for LSTM and Transformer...')
X_seq_train, y_seq_train = create_sequences_from_static(X_train, y_train)
X_seq_val, y_seq_val = create_sequences_from_static(X_val, y_val)
X_seq_test, y_seq_test = create_sequences_from_static(X_test, y_test)

y_seq_train_oh = keras.utils.to_categorical(y_seq_train, NUM_ACTUAL_CLASSES)
y_seq_val_oh = keras.utils.to_categorical(y_seq_val, NUM_ACTUAL_CLASSES)
y_seq_test_oh = keras.utils.to_categorical(y_seq_test, NUM_ACTUAL_CLASSES)

print('Sequence shape:', X_seq_train.shape)
print('Sequence data ready')

## Cell 10: Model 3 - LSTM Temporal Classifier

In [ ]:
def build_lstm(seq_len, feature_dim, num_classes):
    inputs = keras.Input(shape=(seq_len, feature_dim), name='sequence')
    x = layers.LSTM(256, return_sequences=True)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(128, return_sequences=True)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(64, return_sequences=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    model = Model(inputs, outputs, name='koe_lstm')
    return model

lstm_model = build_lstm(SEQ_LEN, 63, NUM_ACTUAL_CLASSES)
lstm_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
lstm_model.summary()

callbacks_lstm = [
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, verbose=1),
    ModelCheckpoint(f'{OUTPUT_DIR}/lstm_best.keras', monitor='val_accuracy',
                   save_best_only=True, verbose=0)
]

print('Training LSTM model...')
history_lstm = lstm_model.fit(
    X_seq_train, y_seq_train_oh,
    validation_data=(X_seq_val, y_seq_val_oh),
    epochs=80,
    batch_size=64,
    callbacks=callbacks_lstm,
    verbose=1
)

lstm_test_loss, lstm_test_acc = lstm_model.evaluate(X_seq_test, y_seq_test_oh, verbose=0)
print(f'LSTM Test Accuracy: {lstm_test_acc:.4f}')
lstm_model.save(f'{OUTPUT_DIR}/lstm_model.keras')
print('LSTM model saved')

## Cell 11: Model 4 - Transformer Classifier

In [ ]:
def positional_encoding(seq_len, d_model):
    positions = np.arange(seq_len)[:, np.newaxis]
    dims = np.arange(d_model)[np.newaxis, :]
    angles = positions / np.power(10000, (2 * (dims // 2)) / d_model)
    angles[:, 0::2] = np.sin(angles[:, 0::2])
    angles[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.cast(angles[np.newaxis, ...], dtype=tf.float32)

class TransformerBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1, **kwargs):
        super().__init__(**kwargs)
        self.att = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation='relu'),
            layers.Dense(embed_dim)
        ])
        self.layernorm1 = layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = layers.Dropout(rate)
        self.dropout2 = layers.Dropout(rate)

    def call(self, inputs, training=False):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

def build_transformer(seq_len, feature_dim, num_classes,
                       embed_dim=128, num_heads=4, ff_dim=256, num_blocks=3):
    inputs = keras.Input(shape=(seq_len, feature_dim), name='sequence')
    x = layers.Dense(embed_dim)(inputs)
    pos_enc = positional_encoding(seq_len, embed_dim)
    x = x + pos_enc
    for _ in range(num_blocks):
        x = TransformerBlock(embed_dim, num_heads, ff_dim, rate=0.1)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)
    model = Model(inputs, outputs, name='koe_transformer')
    return model

transformer_model = build_transformer(SEQ_LEN, 63, NUM_ACTUAL_CLASSES)
transformer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
transformer_model.summary()

callbacks_transformer = [
    EarlyStopping(monitor='val_accuracy', patience=12, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, verbose=1),
    ModelCheckpoint(f'{OUTPUT_DIR}/transformer_best.keras', monitor='val_accuracy',
                   save_best_only=True, verbose=0)
]

print('Training Transformer model...')
history_transformer = transformer_model.fit(
    X_seq_train, y_seq_train_oh,
    validation_data=(X_seq_val, y_seq_val_oh),
    epochs=80,
    batch_size=64,
    callbacks=callbacks_transformer,
    verbose=1
)

trans_test_loss, trans_test_acc = transformer_model.evaluate(X_seq_test, y_seq_test_oh, verbose=0)
print(f'Transformer Test Accuracy: {trans_test_acc:.4f}')
transformer_model.save(f'{OUTPUT_DIR}/transformer_model.keras')
print('Transformer model saved')

## Cell 12: Model 5 - Ensemble Meta-Classifier

In [ ]:
print('Generating ensemble features from all models...')

# Get predictions from all models on validation set
mlp_val_preds = mlp_model.predict(X_val, verbose=0)
cnn_val_preds = cnn_model.predict(X_heatmap_val, verbose=0)
lstm_val_preds = lstm_model.predict(X_seq_val, verbose=0)
trans_val_preds = transformer_model.predict(X_seq_val, verbose=0)

# Concatenate all predictions as meta-features
meta_X_val = np.concatenate([
    mlp_val_preds,
    cnn_val_preds,
    lstm_val_preds,
    trans_val_preds
], axis=1)
print('Meta feature shape:', meta_X_val.shape)

# Get test predictions for evaluation
mlp_test_preds = mlp_model.predict(X_test, verbose=0)
cnn_test_preds = cnn_model.predict(X_heatmap_test, verbose=0)
lstm_test_preds = lstm_model.predict(X_seq_test, verbose=0)
trans_test_preds = transformer_model.predict(X_seq_test, verbose=0)

meta_X_test = np.concatenate([
    mlp_test_preds,
    cnn_test_preds,
    lstm_test_preds,
    trans_test_preds
], axis=1)

# Build meta-classifier
def build_ensemble_meta(input_dim, num_classes):
    inputs = keras.Input(shape=(input_dim,), name='ensemble_input')
    x = layers.Dense(256, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    x = layers.Dense(64, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='final_prediction')(x)
    model = Model(inputs, outputs, name='koe_ensemble')
    return model

meta_input_dim = meta_X_val.shape[1]
ensemble_model = build_ensemble_meta(meta_input_dim, NUM_ACTUAL_CLASSES)
ensemble_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Training ensemble meta-classifier...')
history_ensemble = ensemble_model.fit(
    meta_X_val, y_val_oh,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[EarlyStopping(patience=10, restore_best_weights=True, verbose=1)],
    verbose=1
)

# Evaluate ensemble on test set
ensemble_test_loss, ensemble_test_acc = ensemble_model.evaluate(
    meta_X_test, y_test_oh, verbose=0
)
print(f'\nEnsemble Test Accuracy: {ensemble_test_acc:.4f}')
ensemble_model.save(f'{OUTPUT_DIR}/ensemble_model.keras')
print('Ensemble model saved')

## Cell 13: Results Summary

In [ ]:
print('=' * 60)
print('KOE MODEL TRAINING RESULTS')
print('=' * 60)
print(f'Total training samples: {len(X_train)}')
print(f'Number of classes: {NUM_ACTUAL_CLASSES}')
print(f'Classes: {list(le.classes_)}')
print()
print('Model Performance:')
print(f'  MLP Classifier:       {mlp_test_acc:.4f} ({mlp_test_acc*100:.1f}%)')
print(f'  CNN Heatmap:          {cnn_test_acc:.4f} ({cnn_test_acc*100:.1f}%)')
print(f'  LSTM Temporal:        {lstm_test_acc:.4f} ({lstm_test_acc*100:.1f}%)')
print(f'  Transformer:          {trans_test_acc:.4f} ({trans_test_acc*100:.1f}%)')
print(f'  Ensemble (Final):     {ensemble_test_acc:.4f} ({ensemble_test_acc*100:.1f}%)')
print()
print('Models saved to:', OUTPUT_DIR)

# Plot training curves
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, history, name in zip(
    axes.flatten(),
    [history_mlp, history_cnn, history_lstm, history_transformer],
    ['MLP', 'CNN', 'LSTM', 'Transformer']
):
    ax.plot(history.history['accuracy'], label='Train', color='#B45309')
    ax.plot(history.history['val_accuracy'], label='Val', color='#78716C')
    ax.set_title(f'{name} Accuracy', fontsize=12, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Accuracy')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Koe Model Training Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Training curves saved')

## Cell 14: Convert All Models to TFLite

In [ ]:
# Cell 14: Convert All Models to TFLite

def convert_to_tflite(model, output_path, quantize=True):
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    if quantize:
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]
    tflite_model = converter.convert()
    with open(output_path, 'wb') as f:
        f.write(tflite_model)
    size_kb = os.path.getsize(output_path) / 1024
    print(f'Saved: {output_path} ({size_kb:.1f} KB)')
    return tflite_model

print('Converting models to TFLite format...')
print()

mlp_tflite = convert_to_tflite(mlp_model, f'{OUTPUT_DIR}/koe_mlp.tflite')
cnn_tflite = convert_to_tflite(cnn_model, f'{OUTPUT_DIR}/koe_cnn.tflite')
print('Skipping LSTM TFLite - GPU CudnnRNNV3 incompatible. LSTM .keras saved for server-side use.')
lstm_model.save(f'{OUTPUT_DIR}/koe_lstm.keras')
transformer_tflite = convert_to_tflite(transformer_model, f'{OUTPUT_DIR}/koe_transformer.tflite')
ensemble_tflite = convert_to_tflite(ensemble_model, f'{OUTPUT_DIR}/koe_ensemble.tflite')

print()
print('All models ready')
print('Files in output directory:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f'  {f}: {size:.1f} KB')


## Cell 15: Save Config and Verify Models

In [ ]:
# Save model config for the Koe backend
config = {
    'num_classes': NUM_ACTUAL_CLASSES,
    'classes': list(le.classes_),
    'koe_signs': KOE_SIGNS,
    'feature_dim': 63,
    'seq_len': SEQ_LEN,
    'models': {
        'mlp': 'koe_mlp.tflite',
        'cnn': 'koe_cnn.tflite',
        'lstm': 'koe_lstm.tflite',
        'transformer': 'koe_transformer.tflite',
        'ensemble': 'koe_ensemble.tflite'
    },
    'test_accuracy': {
        'mlp': float(mlp_test_acc),
        'cnn': float(cnn_test_acc),
        'lstm': float(lstm_test_acc),
        'transformer': float(trans_test_acc),
        'ensemble': float(ensemble_test_acc)
    }
}

with open(f'{OUTPUT_DIR}/koe_model_config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Config saved')

# Quick inference test to verify all models work
print('\nRunning inference verification...')
test_landmark = X_test[0:1]
test_heatmap = X_heatmap_test[0:1]
test_seq = X_seq_test[0:1]

mlp_pred = mlp_model.predict(test_landmark, verbose=0)
cnn_pred = cnn_model.predict(test_heatmap, verbose=0)
lstm_pred = lstm_model.predict(test_seq, verbose=0)
trans_pred = transformer_model.predict(test_seq, verbose=0)
meta_input = np.concatenate([mlp_pred, cnn_pred, lstm_pred, trans_pred], axis=1)
final_pred = ensemble_model.predict(meta_input, verbose=0)

predicted_class = le.classes_[np.argmax(final_pred[0])]
true_class = le.classes_[y_test[0]]
confidence = float(np.max(final_pred[0]))

print(f'True class: {true_class}')
print(f'Predicted:  {predicted_class}')
print(f'Confidence: {confidence:.4f}')
print()
print('=' * 60)
print('ALL KOE MODELS READY')
print('=' * 60)
print(f'Download your models from: /kaggle/working/koe_models/')
print(f'Final ensemble accuracy: {ensemble_test_acc*100:.1f}%')
print('Upload these files to Google Cloud Storage for the Koe backend')